# Práctica 2: Aprendizaje no supervisado
## Determinación de tipos de estrellas

**Asignatura:** Aprendizaje Automático  
**Grupo de prácticas:** [X]  
**Integrantes:** [Nombre 1, Nombre 2]  
**Semilla base utilizada:** [NIA o nº de grupo]

## 1. Objetivo

El objetivo de esta práctica es aplicar técnicas de aprendizaje no supervisado para
agrupar estrellas en función de sus características físicas y espectrales.

Para ello, se parte de un conjunto de datos de 240 estrellas y se comparan tres
algoritmos de clustering:
- K-Means
- Clustering jerárquico
- DBSCAN

Siguiendo el enunciado, antes del clustering se reduce la dimensionalidad a 2
componentes principales mediante PCA. Finalmente, se recomienda un pipeline de
clustering y se compara con las clases astronómicas habituales.


In [3]:
# Instalación de dependencias necesarias
%pip install numpy pandas matplotlib seaborn scikit-learn scipy

  Using cached matplotlib-3.10.8-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached contourpy-1.3.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 38.8 MB/s eta 0:00:00m eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 61.3 MB/s eta 0:00:00
Using cached matplotl

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from scipy.cluster.hierarchy import linkage, dendrogram

SEED = 123456  # sustituir por NIA o nº de grupo
np.random.seed(SEED)

En esta práctica se fija una semilla para garantizar la reproducibilidad de los
resultados, tal y como exige el enunciado.

In [5]:
df = pd.read_csv("stars_data.csv")  # revisar nombre exacto del fichero
df.head()

,Temperature,L,R,A_M,Color,Spectral_Class
0,3068,0.002400,0.1700,16.12,Red,M
1,3042,0.000500,0.1542,16.60,Red,M
2,2600,0.000300,0.1020,18.70,Red,M
3,2800,0.000200,0.1600,16.65,Red,M
4,1939,0.000138,0.1030,20.06,Red,M


In [6]:
df.shape, df.info()

<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Temperature     240 non-null    int64  
 1   L               240 non-null    float64
 2   R               240 non-null    float64
 3   A_M             240 non-null    float64
 4   Color           240 non-null    str    
 5   Spectral_Class  240 non-null    str    
dtypes: float64(3), int64(1), str(2)
memory usage: 11.4 KB


((240, 6), None)

In [7]:
df.describe(include='all')

,Temperature,L,R,A_M,Color,Spectral_Class
count,240.000000,240.000000,240.000000,240.000000,240,240
unique,NaN,NaN,NaN,NaN,17,7
top,NaN,NaN,NaN,NaN,Red,M
freq,NaN,NaN,NaN,NaN,112,111
mean,10497.462500,107188.361635,237.157781,4.382396,NaN,NaN
std,9552.425037,179432.244940,517.155763,10.532512,NaN,NaN
min,1939.000000,0.000080,0.008400,-11.920000,NaN,NaN
25%,3344.250000,0.000865,0.102750,-6.232500,NaN,NaN
50%,5776.000000,0.070500,0.762500,8.313000,NaN,NaN
75%,15055.500000,198050.000000,42.750000,13.697500,NaN,NaN


## 3. Descripción inicial del dataset

El conjunto de datos contiene 240 estrellas y las siguientes variables:
- Temperature
- L
- R
- A_M
- Spectral_Class
- Color

A continuación se revisan tipos de datos, posibles valores categóricos y la existencia
de valores perdidos o inconsistencias.